# Baseline Experiments

6 fusion methods x 7 modality combos x 3 seeds x 2 splits = **252 runs**

Disconnect-safe: re-run Cell 3 to resume from where it left off.

In [1]:
# Cell 1: Mount Drive + setup
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyyaml', 'scipy', '-q'])

%cd /content/drive/MyDrive/ProjectExperiment
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Mounted at /content/drive
/content/drive/MyDrive/ProjectExperiment
NVIDIA A100-SXM4-80GB, 81920 MiB


In [2]:
!grep -c "workers" scripts/run_experiment.py

22


In [3]:
# Cell 2: Dry run (preview plan, no training)
!python scripts/run_experiment.py \
    --sweep configs/sweeps/full_ablation.yaml \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/baselines \
    --dry_run 2>&1 | tail -5

!echo "---"
!python scripts/run_experiment.py \
    --sweep configs/sweeps/full_ablation.yaml \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/baselines \
    --dry_run 2>&1 | grep -c '\[RUN\]'

  [RUN] gated_state_trend / dual_km_telem / seed1
  [RUN] gated_state_trend / dual_km_telem / seed2
  [RUN] gated_state_trend / triple_video_km_telem / seed0
  [RUN] gated_state_trend / triple_video_km_telem / seed1
  [RUN] gated_state_trend / triple_video_km_telem / seed2
---
87


In [5]:
# Cell 3: Run experiments (re-run this cell to resume after disconnect)
!python scripts/run_experiment.py \
    --sweep configs/sweeps/full_ablation.yaml \
    --workers 1 \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/baselines

Sweep plan: 252 runs total, 251 already completed, 1 to run

[1/252] SKIP late_state_trend/single_video/seed0 (already done)
[2/252] SKIP late_state_trend/single_video/seed1 (already done)
[3/252] SKIP late_state_trend/single_video/seed2 (already done)
[4/252] SKIP late_state_trend/single_km/seed0 (already done)
[5/252] SKIP late_state_trend/single_km/seed1 (already done)
[6/252] SKIP late_state_trend/single_km/seed2 (already done)
[7/252] SKIP late_state_trend/single_telem/seed0 (already done)
[8/252] SKIP late_state_trend/single_telem/seed1 (already done)
[9/252] SKIP late_state_trend/single_telem/seed2 (already done)
[10/252] SKIP late_state_trend/dual_video_km/seed0 (already done)
[11/252] SKIP late_state_trend/dual_video_km/seed1 (already done)
[12/252] SKIP late_state_trend/dual_video_km/seed2 (already done)
[13/252] SKIP late_state_trend/dual_video_telem/seed0 (already done)
[14/252] SKIP late_state_trend/dual_video_telem/seed1 (already done)
[15/252] SKIP late_state_trend/dual_

In [6]:
# Cell 4: Check progress (run anytime)
import glob

RUNS_ROOT = '/content/drive/MyDrive/AmuCS_experiment/runs/baselines'
done = glob.glob(f'{RUNS_ROOT}/**/metrics.json', recursive=True)
print(f'Completed: {len(done)} / 252')

# Per-task breakdown
from collections import Counter
tasks = Counter()
for p in done:
    parts = p.replace('\\', '/').split('/')
    for part in parts:
        if '_3seed' in part:
            tasks[part.replace('_3seed', '')] += 1
            break

for task, count in sorted(tasks.items()):
    total = 7 * 3  # 7 modality combos x 3 seeds
    print(f'  {task}: {count}/{total}')

Completed: 252 / 252
  cma_state_trend: 42/21
  eft_state_trend: 42/21
  gated_state_trend: 42/21
  late_state_trend: 42/21
  lft_state_trend: 42/21
  mft_state_trend: 42/21
